# Semantic segmentation — U-Net

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. Segmentation problem
A segmentation model predicts a class per pixel. This tutorial defines a compact U-Net and a dataset interface. Use paired image/mask files with identical geometry.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from torch import nn
print(torch.__version__)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Dataset contract
Images are float tensors [3,H,W]; masks are integer tensors [H,W] with class IDs 0..C-1. Do not normalize masks or apply image-only geometric transforms.

In [ ]:
from torch.utils.data import Dataset
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, image_transform=None):
        assert len(image_paths)==len(mask_paths)
        self.images,self.masks=image_paths,mask_paths
        self.image_transform=image_transform
    def __len__(self): return len(self.images)
    def __getitem__(self,i):
        # TODO: load RGB image and single-channel mask with PIL/OpenCV.
        # Apply the SAME resize/crop/flip to image and mask; use nearest-neighbor for masks.
        raise NotImplementedError('Connect your paired image/mask loader here')

## 3. U-Net blocks
Skip connections preserve spatial details lost during downsampling.

In [ ]:
def conv_block(cin,cout):
    return nn.Sequential(nn.Conv2d(cin,cout,3,padding=1,bias=False),nn.BatchNorm2d(cout),nn.ReLU(inplace=True),nn.Conv2d(cout,cout,3,padding=1,bias=False),nn.BatchNorm2d(cout),nn.ReLU(inplace=True))
class UNet(nn.Module):
    def __init__(self,n_classes=2):
        super().__init__(); self.e1=conv_block(3,32); self.e2=conv_block(32,64); self.e3=conv_block(64,128); self.pool=nn.MaxPool2d(2)
        self.mid=conv_block(128,256); self.up3=nn.ConvTranspose2d(256,128,2,2); self.d3=conv_block(256,128); self.up2=nn.ConvTranspose2d(128,64,2,2); self.d2=conv_block(128,64); self.up1=nn.ConvTranspose2d(64,32,2,2); self.d1=conv_block(64,32); self.out=nn.Conv2d(32,n_classes,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b)); z=self.mid(self.pool(c))
        z=self.d3(torch.cat([self.up3(z),c],1)); z=self.d2(torch.cat([self.up2(z),b],1)); z=self.d1(torch.cat([self.up1(z),a],1)); return self.out(z)
model=UNet(n_classes=2).to(device)
print(model(torch.randn(2,3,128,128,device=device)).shape)

## 4. Loss and metrics
For mutually exclusive classes use cross-entropy; Dice loss can help with imbalance. Compute metrics on the original class IDs.

In [ ]:
def dice_per_class(logits, target, n_classes, eps=1e-7):
    pred=logits.argmax(1); scores=[]
    for c in range(n_classes):
        p=(pred==c); t=(target==c); scores.append((2*(p&t).sum().float()+eps)/(p.sum()+t.sum()+eps))
    return torch.stack(scores)
criterion=nn.CrossEntropyLoss()
# Training step: logits=model(images); loss=criterion(logits,masks.long())
# optimizer.zero_grad(); loss.backward(); optimizer.step()

## 5. TensorFlow/Keras alternative
Keras offers `Conv2D`, `MaxPooling2D`, `UpSampling2D` and `Concatenate` for the same encoder-decoder design. Use sparse categorical cross-entropy for integer masks; use nearest-neighbor resize for masks.

In [ ]:
import tensorflow as tf
print('TensorFlow:',tf.__version__)
# For a complete Keras port, mirror the same skip connections using Functional API.
# Compile with SparseCategoricalCrossentropy(from_logits=True) when final layer has no softmax.

## 6. Evaluation
Report mean IoU, per-class Dice/IoU, pixel accuracy and boundary quality. State whether background is included in the mean.

## Exercises
Complete the dataset loader; train on Oxford-IIIT Pet or a permitted custom dataset; add class weights; compare U-Net with DeepLabV3; test mixed precision and tiled inference.